<div dir="rtl">
  <p style="color: #80f9fa; text-align: right;">
    <b>بلوک squeeze and excitation</b>
  </p>
</div>

---

In [6]:
import torch
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)  # Squeeze
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)       # Squeeze
        y = self.fc(y).view(b, c, 1, 1)       # Excitation
        return x * y.expand_as(x)             # Scale channels


model = SEBlock(channel=3, reduction=16)
x = torch.randn(1, 3, 32, 32)
y = model(x)
print(y.shape)

e:\Computer-Vision-using-Deep-Learning\.venv\Lib\site-packages\torch\nn\init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


torch.Size([1, 3, 32, 32])


In [8]:
import torch
class SEBlock(torch.nn.Module):
    def __init__(self, channel, reduction, H, W):
        super(SEBlock, self).__init__()
        self.squeeze = torch.nn.AvgPool2d(kernel_size=(H, W))
        self.excitation = torch.nn.Sequential(
            torch.nn.Linear(channel, channel // reduction, bias=False),
            torch.nn.ReLU(),
            torch.nn.Linear(channel // reduction, channel, bias=False),
            torch.nn.Sigmoid()
        )
        self.flatten = torch.nn.Flatten()

    def forward(self, x):
        b, c, h, w = x.size()
        x_squeeze = self.squeeze(x).view(b, c)
        x_excitation = self.excitation(self.flatten(x_squeeze)).view(b, c, 1, 1)
        return x_excitation.expand_as(x) * x

model = SEBlock(channel=3, reduction=16, H=32, W=32)
x = torch.randn(1, 3, 32, 32)
y = model(x)
print(y.shape)

e:\Computer-Vision-using-Deep-Learning\.venv\Lib\site-packages\torch\nn\init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


torch.Size([1, 3, 32, 32])


---

<div dir="rtl" style="text-align: right;">
ماژول
SE
در
torchvision
</div>

```python
torchvision.ops.SqueezeExcitation
```

- input_channels (int) – Number of channels in the input image

- squeeze_channels (int) – Number of squeeze channels

- activation (Callable[..., torch.nn.Module], optional) – delta activation. Default: torch.nn.ReLU

- scale_activation (Callable[..., torch.nn.Module]) – sigma activation. Default: torch.nn.Sigmoid

In [ ]:
from typing import Callable

import torch
from torch import Tensor

class SqueezeExcitation(torch.nn.Module):
    """
    This block implements the Squeeze-and-Excitation block from https://arxiv.org/abs/1709.01507 (see Fig. 1).
    Parameters ``activation``, and ``scale_activation`` correspond to ``delta`` and ``sigma`` in eq. 3.

    Args:
        input_channels (int): Number of channels in the input image
        squeeze_channels (int): Number of squeeze channels
        activation (Callable[..., torch.nn.Module], optional): ``delta`` activation. Default: ``torch.nn.ReLU``
        scale_activation (Callable[..., torch.nn.Module]): ``sigma`` activation. Default: ``torch.nn.Sigmoid``
    """

    def __init__(
        self,
        input_channels: int,
        squeeze_channels: int,
        activation: Callable[..., torch.nn.Module] = torch.nn.ReLU,
        scale_activation: Callable[..., torch.nn.Module] = torch.nn.Sigmoid,
    ) -> None:
        super().__init__()
        self.avgpool = torch.nn.AdaptiveAvgPool2d(1)
        self.fc1 = torch.nn.Conv2d(input_channels, squeeze_channels, 1)
        self.fc2 = torch.nn.Conv2d(squeeze_channels, input_channels, 1)
        self.activation = activation()
        self.scale_activation = scale_activation()

    def _scale(self, input: Tensor) -> Tensor:
        scale = self.avgpool(input)
        scale = self.fc1(scale)
        scale = self.activation(scale)
        scale = self.fc2(scale)
        return self.scale_activation(scale)
    
    def forward(self, input: Tensor) -> Tensor:
        scale = self._scale(input)
        return scale * input

In [2]:
import torch
from torchvision.ops import SqueezeExcitation

input_channels = 3
reduction = 16

se_block = SqueezeExcitation(input_channels, reduction)

x = torch.randn(1, 3, 32, 32)

output = se_block(x)

print(output.shape)

torch.Size([1, 3, 32, 32])


---

<div dir="rtl" style="text-align: right;">
تمرین:
یک شبکه عصبی کانولوشنی برای داده‌ی
mnist
براساس معماری زیر و مطابق با قالب پیوست، آموزش بدهید.
لازم به ذکر است، کانولوشن اول به‌کاررفته در بلوک
residual
را با نسخه‌ی
SE
جایگزین نمایید.

معماری پیشنهادی

- بلوک Sequential:
    - Conv2D(32, 3×3, padding=1) → ReLU → MaxPool(2×2)

- (Residual Block):
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(64, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU

- لایه‌های Fully Connected (Dense) در مدل Sequential:
    - Flatten()
    - Dense(512) → ReLU → Dropout(0.4)
    - Dense(256) → ReLU → Dropout(0.4)
    - Dense(128) → ReLU
    - Dense(10)
</div>

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)  # Squeeze
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)       # Squeeze
        y = self.fc(y).view(b, c, 1, 1)       # Excitation
        return x * y.expand_as(x)             # Scale channels

class Model(nn.Module):
    def __init__(self, num_classes=10):
        super(Model, self).__init__()
        

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )


        self.conv_block2 = nn.Sequential(
            # nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding="same"),
            SEBlock(channel=32),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, padding="same"),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )


        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = x + self.conv_block2(x)
        x = self.fc_block(x)
        return x

x = torch.randn(1, 1, 28, 28)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 10])
